# BEH 10: customer and policy lifecycle

The spine of the rolling customer behaviour dataset: one row per customer, policy and month-end
(`beh.CustomerPolicyMonth`) with the policy's position in its policy year and the outcome of its upcoming renewal,
and one row per customer and month-end (`beh.CustomerMonth`) with holdings and tenure.

Month-ends run from `hist_start` (history for the 24-month windows) to `last_me`; `IsAnalysisMonth` marks the
analysis months (from `first_obs`). A customer is in a month when they hold an in-force (CloseNum = 1) Motor, Home or
Van policy at the month-end, Temporary (T) policies excluded. Lifecycle: months to renewal = calendar months from the
month-end to the renewal month, policy month = 13 minus that (M1 = 12 months to go, M11 = 2, M12 = the renewal month
before due). Retention = the ods_v2.PolicyRenewal offered / retained flags as they stand (clarifications D1).

**If a cell fails**

1. Read the description above the failing cell: what it builds, what it reads, its grain and key.
2. Fix the cause (usually a renamed or missing source column, or a broken key), then rerun from that cell.
   Every cell is `create or replace`, so rerunning is safe.
3. The checks cell at the end names any table that is empty or has duplicate / null keys, and the likely cause.

Run order: BEH_00_Setup is run by every notebook. Then 10, 20, 30, 40, 50, 60, in that order.


In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "ChillLH"
    }
}

In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run BEH_00_Setup

In [ ]:
%%pyspark
beh_run_log("BEH_10_Lifecycle", "started")


## beh_month_end


**`stg.beh_month_end`**

The month-ends the dataset is built for, with a month number for window arithmetic (MonthNo = year \* 12 + month) and the analysis-period flag.

- **Reads:** settings hist_start, first_obs, last_me
- **Key:** MonthEndDate

In [ ]:
create or replace table ${beh.stg}.beh_month_end as
select  MonthEndDate,
        year(MonthEndDate) * 12 + month(MonthEndDate)                              as MonthNo,
        case when MonthEndDate >= to_date('${beh.first_obs}') then 1 else 0 end    as IsAnalysisMonth
from    (select last_day(add_months(to_date('${beh.hist_start}'), i)) as MonthEndDate
         from   (select explode(sequence(0, cast(round(months_between(to_date('${beh.last_me}'),
                                                                       to_date('${beh.hist_start}'))) as int))) as i) s) d


## beh_policy_scv


**`stg.beh_policy_scv`**

Relay policy code to SCV customer key. ods.scveadmlinkfinal is the only policy link (house rule); one key per code in Oct 2026 (checked: 0 codes on two keys). Lowest key taken if that changes; ScvCandidates says how many there were.

- **Reads:** ods.scveadmlinkfinal (SourceSystemId 1 = Relay policy)
- **Key:** PolicyCode

In [ ]:
create or replace table ${beh.stg}.beh_policy_scv as
select  trim(ChillSourcePolicyReference)    as PolicyCode,
        min(SCV_Customer_Key)               as SCV_Customer_Key,
        count(distinct SCV_Customer_Key)    as ScvCandidates
from    ${beh.link}.scveadmlinkfinal
where   SourceSystemId = 1
  and   SCV_Customer_Key is not null
  and   nullif(trim(ChillSourcePolicyReference), '') is not null
group by trim(ChillSourcePolicyReference)


## beh_customer_since


**`stg.beh_customer_since`**

The date each customer first incepted a Motor, Home or Van policy (any code, any time), for tenure. PolicyMonthEnd.ClientTenure is not a plain month count, so tenure is derived here.

- **Reads:** ods_v2.Policy, stg.beh_policy_scv
- **Key:** SCV_Customer_Key

In [ ]:
create or replace table ${beh.stg}.beh_customer_since as
select  l.SCV_Customer_Key,
        min(p.PolicyInceptDate)     as CustomerSinceDate
from    ${beh.ods}.policy p
join    ${beh.stg}.beh_policy_scv l on l.PolicyCode = p.PolicyCode
where   p.PolicyTypeGroup in ('Motor', 'Home', 'Van')
  and   coalesce(p.PolicyStatusCode, '') <> 'T'
  and   p.PolicyInceptDate >= date'1990-01-01'
group by l.SCV_Customer_Key


## beh_motor_addon


**`stg.beh_motor_addon`**

Add-ons on each in-force Motor policy at each month-end (curation #13): KeyCare on the latest sale row of the policy on or before the month-end (PolicySale.KeyCareFlag, the warehouse kci_flag, recorded from 2023: null for a term sold earlier) and NCD protection on the proposer version valid at the month-end: NcdProtected = 1 for Full protection only, because Step Back is on about 70% of proposer versions (the standard terms, not an add-on). The level is kept as NcdProtectionLevel.

- **Reads:** ods_v2.PolicyMonthEnd, ods_v2.PolicySale, ods_v2.PolicyMotorProposer, stg.beh_month_end
- **Key:** PolicyCode + MonthEndDate

In [ ]:
create or replace table ${beh.stg}.beh_motor_addon as
select  k.PolicyCode,
        k.MonthEndDate,
        k.KeyCareHeld,
        max(case when v.ProposerNCDProtection = 'Full' then 1
                 when v.ProposerNCDProtection is not null then 0 end)          as NcdProtected,
        max(v.ProposerNCDProtection)                                        as NcdProtectionLevel
from    (select  PolicyCode,
                 MonthEndDate,
                 case when SaleDate is null or SaleDate < date'2023-01-01' then null
                      when KeyCareFlag = 1 then 1 else 0 end                   as KeyCareHeld
         from    (select  p.PolicyCode,
                          m.MonthEndDate,
                          s.ReportingSaleDate                                  as SaleDate,
                          s.KeyCareFlag,
                          row_number() over (partition by p.PolicyCode, m.MonthEndDate
                                             order by s.ReportingSaleDate desc) as rn
                  from    ${beh.ods}.policymonthend p
                  join    ${beh.stg}.beh_month_end m on m.MonthEndDate = p.EffectiveDate
                  left join ${beh.ods}.policysale s
                          on  s.PolicyCode = p.PolicyCode
                          and s.ReportingSaleDate <= m.MonthEndDate
                          and s.ReportingSaleType in ('New Business', 'Renewals')
                          and s.SaleCount > 0
                  where   p.CloseNum = 1
                    and   p.PolicyTypeGroup = 'Motor') x
         where   rn = 1) k
left join ${beh.ods}.policymotorproposer v
        on  v.PolicyCode = k.PolicyCode
        and v.RecordDateTimeStart <= cast(date_add(k.MonthEndDate, 1) as timestamp)
        and (v.RecordDateTimeEnd is null or v.RecordDateTimeEnd > cast(date_add(k.MonthEndDate, 1) as timestamp))
group by k.PolicyCode, k.MonthEndDate, k.KeyCareHeld


## beh_policy_latest


**`stg.beh_policy_latest`**

The latest version of every policy (ods_v2.Policy, RecordVersionIsLatest): status, inception, renewal date and premium as they stand today, for the renewal signals in stg.beh_motor_retention.

- **Reads:** ods_v2.Policy
- **Key:** PolicyCode

In [ ]:
create or replace table ${beh.stg}.beh_policy_latest as
select  PolicyCode,
        PolicyTypeGroup,
        PolicyStatusDesc,
        to_date(PolicyInceptDate)       as PolicyInceptDate,
        to_date(PolicyRenewDate)        as PolicyRenewDate,
        CCYGrossPremium
from    ${beh.ods}.policy
where   RecordVersionIsLatest = true


## beh_motor_retention


**`stg.beh_motor_retention`**

Motor renewal retained (agreed with Vijay 3 Nov 2026, run note section 13): offered Motor renewals only, retained when any of four signals fires, read in this order (RetentionSignal names the first): 1 the warehouse flag (PolicyRetNum); 2 the same policy rolled on (status New Business or Renewal and a renewal date more than 300 days after the due); 3 the same SCV customer took a priced Motor policy (New Business or Renewal, the same car or another) under a different code, incepted from 60 days before to 30 days after the due, that is not already the renewal of a different policy (the warehouse's own guard: one new policy renews one old policy); 4 the renewals fact holds a row for the same policy a year on. The structural segmentation rule (Chill_structural_pack sc_offers.py), with signal 3 on the SCV key, the 60 / 30 day window and the guard. Definitions and reasons: out/renewal_definition.md.

- **Reads:** ods_v2.PolicyRenewal, stg.beh_policy_latest, stg.beh_policy_scv
- **Key:** PolicyCode + DueDate

In [ ]:
create or replace table ${beh.stg}.beh_motor_retention as
select  r.PolicyCode,
        r.LYPolicyRenewDateAdj                                                      as DueDate,
        case when coalesce(r.PolicyRetNum, 0) > 0 then '1 warehouse flag'
             when p.PolicyStatusDesc in ('New Business', 'Renewal')
              and p.PolicyRenewDate > date_add(r.LYPolicyRenewDateAdj, 300) then '2 same policy rolled on'
             when s3.PolicyCode is not null then '3 same customer, new Motor policy'
             when s4.PolicyCode is not null then '4 renewals fact row a year on'
        end                                                                         as RetentionSignal
from    ${beh.ods}.policyrenewal r
left join ${beh.stg}.beh_policy_latest p
        on  p.PolicyCode = r.PolicyCode
left join (select  distinct r2.PolicyCode, r2.LYPolicyRenewDateAdj
           from    ${beh.ods}.policyrenewal r2
           join    ${beh.stg}.beh_policy_scv l2 on l2.PolicyCode = r2.PolicyCode
           join    ${beh.stg}.beh_policy_scv lk on lk.SCV_Customer_Key = l2.SCV_Customer_Key
           join    ${beh.stg}.beh_policy_latest pk
                   on  pk.PolicyCode = lk.PolicyCode
                   and pk.PolicyTypeGroup = 'Motor'
                   and pk.PolicyStatusDesc in ('New Business', 'Renewal')
                   and pk.CCYGrossPremium > 0
           left join (select  distinct upper(trim(coalesce(TYPolicyCode, NewPolicyCode))) as RenewedTo,
                              PolicyCode                                                    as RenewedFrom
                      from    ${beh.ods}.policyrenewal
                      where   LYPolicyTypeGroup = 'Motor'
                        and   coalesce(TYPolicyCode, NewPolicyCode) is not null) o
                   on  o.RenewedTo = upper(trim(pk.PolicyCode))
                   and o.RenewedFrom <> r2.PolicyCode
                   and upper(trim(o.RenewedFrom)) <> upper(trim(pk.PolicyCode))
           where   r2.LYPolicyTypeGroup = 'Motor'
             and   r2.PolicyOfferNum = 1
             and   o.RenewedTo is null
             and   pk.PolicyCode <> r2.PolicyCode
             and   pk.PolicyInceptDate between date_sub(r2.LYPolicyRenewDateAdj, 60)
                                           and date_add(r2.LYPolicyRenewDateAdj, 30)) s3
        on  s3.PolicyCode = r.PolicyCode
        and s3.LYPolicyRenewDateAdj = r.LYPolicyRenewDateAdj
left join (select  distinct r3.PolicyCode, r3.LYPolicyRenewDateAdj
           from    ${beh.ods}.policyrenewal r3
           join    ${beh.ods}.policyrenewal n
                   on  n.PolicyCode = r3.PolicyCode
                   and trunc(n.RenewalMonth, 'MM') = trunc(add_months(r3.RenewalMonth, 12), 'MM')
           where   r3.LYPolicyTypeGroup = 'Motor'
             and   r3.PolicyOfferNum = 1) s4
        on  s4.PolicyCode = r.PolicyCode
        and s4.LYPolicyRenewDateAdj = r.LYPolicyRenewDateAdj
where   r.LYPolicyTypeGroup = 'Motor'
  and   r.PolicyOfferNum = 1


## CustomerPolicyMonth


**`beh.CustomerPolicyMonth`**

One row per in-force Motor, Home or Van policy and month-end, with its customer, its position in the policy year (dates kept beside the labels) and the outcome of its upcoming renewal. Renewal\* outcome columns are AFTER the month-end: never use them as behaviour inputs. RenewalChannel / SubChannel are the channel the renewal was actually processed through (cost-to-serve track 2).

- **Reads:** ods_v2.PolicyMonthEnd (as-at state and renewal date), ods_v2.PolicyRenewal (outcome), stg.beh_policy_scv, stg.beh_month_end, stg.beh_motor_addon, stg.beh_motor_retention
- **Grain:** policy x month-end (a policy has one customer)
- **Key:** PolicyCode + MonthEndDate

In [ ]:
create or replace table ${beh.target}.CustomerPolicyMonth as
select  x.SCV_Customer_Key,
        x.PolicyCode,
        x.PolicyId,
        x.ClientCode,
        x.PolicyTypeGroup,
        x.MonthEndDate,
        x.MonthNo,
        x.IsAnalysisMonth,
        x.PolicyStatusCode,
        x.PolicyStatusDesc,
        x.ReportingSaleType,
        x.PolicyInceptDate,
        x.RenewalDate,
        add_months(x.RenewalDate, -12)                                          as TermStartDate,
        x.MonthsToRenewal,
        13 - x.MonthsToRenewal                                                  as PolicyMonth,
        datediff(x.RenewalDate, x.MonthEndDate)                                 as DaysToRenewal,
        case when x.MonthsToRenewal between 2 and 12 then 1 else 0 end          as IsLifecycleObservation,
        case when x.MonthsToRenewal <= 1 then 1 else 0 end                      as InRenewalWindow,
        x.EURGrossPremClose,
        x.DiaryPaymentTypeTY,
        x.RiskProfileBand,
        x.VehicleCoverType,
        x.ChannelGroup,
        x.ClientAgeBand,
        a.KeyCareHeld,
        a.NcdProtected,
        a.NcdProtectionLevel,
        case when r.PolicyCode is not null then coalesce(r.PolicyOfferNum, 0) end as RenewalOffered,
        case when r.PolicyOfferNum = 1 and x.RenewalDate <= to_date('${beh.renewal_outcome_end}')
             then case when x.PolicyTypeGroup = 'Motor'
                       then case when t.RetentionSignal is not null then 1 else 0 end
                       else coalesce(r.PolicyRetNum, 0) end end                  as RenewalRetained,
        case when r.PolicyOfferNum = 1 and x.RenewalDate <= to_date('${beh.renewal_outcome_end}')
              and x.PolicyTypeGroup = 'Motor' then t.RetentionSignal end         as RenewalRetentionSignal,
        case when x.RenewalDate <= to_date('${beh.renewal_outcome_end}') then 1 else 0 end
                                                                                as RenewalOutcomeObservable,
        r.TYPolicyCode                                                          as RenewalTYPolicyCode,
        r.Channel                                                               as RenewalChannel,
        r.SubChannel                                                            as RenewalSubChannel,
        r.RenewalsPortal                                                        as RenewalPortal
from    (select  l.SCV_Customer_Key,
                 p.PolicyCode,
                 p.PolicyId,
                 p.ClientCode,
                 p.PolicyTypeGroup,
                 m.MonthEndDate,
                 m.MonthNo,
                 m.IsAnalysisMonth,
                 p.PolicyStatusCode,
                 p.PolicyStatusDesc,
                 p.ReportingSaleType,
                 p.PolicyInceptDate,
                 p.PolicyRenewDateAdj                                               as RenewalDate,
                 year(p.PolicyRenewDateAdj) * 12 + month(p.PolicyRenewDateAdj) - m.MonthNo
                                                                                    as MonthsToRenewal,
                 p.EURGrossPremClose,
                 p.DiaryPaymentTypeTY,
                 p.RiskProfileBand,
                 p.VehicleCoverType,
                 p.ChannelGroup,
                 p.ClientAgeBand
         from    ${beh.ods}.policymonthend p
         join    ${beh.stg}.beh_month_end m   on m.MonthEndDate = p.EffectiveDate
         join    ${beh.stg}.beh_policy_scv l  on l.PolicyCode = p.PolicyCode
         where   p.CloseNum = 1
           and   p.PolicyTypeGroup in ('Motor', 'Home', 'Van')
           and   coalesce(p.PolicyStatusCode, '') <> 'T') x
left join ${beh.ods}.policyrenewal r
        on  r.PolicyCode = x.PolicyCode
        and r.LYPolicyRenewDateAdj = x.RenewalDate
left join ${beh.stg}.beh_motor_retention t
        on  t.PolicyCode = x.PolicyCode
        and t.DueDate = x.RenewalDate
left join ${beh.stg}.beh_motor_addon a
        on  a.PolicyCode = x.PolicyCode
        and a.MonthEndDate = x.MonthEndDate


## CustomerMonth


**`beh.CustomerMonth`**

One row per customer and month-end at which they hold an in-force Motor, Home or Van policy: holdings, tenure, premium and the nearest upcoming renewal (Motor and any product). The base for the behaviour, renewal-history and outcome tables.

- **Reads:** beh.CustomerPolicyMonth, stg.beh_customer_since
- **Grain:** customer x month-end
- **Key:** SCV_Customer_Key + MonthEndDate

In [ ]:
create or replace table ${beh.target}.CustomerMonth as
select  c.SCV_Customer_Key,
        c.MonthEndDate,
        c.MonthNo,
        c.IsAnalysisMonth,
        c.PoliciesHeld,
        c.MotorPolicies,
        c.HomePolicies,
        c.VanPolicies,
        c.HoldsMotor,
        c.HoldsHome,
        c.HoldsVan,
        c.HoldsMotor + c.HoldsHome + c.HoldsVan                                as ProductsHeld,
        s.CustomerSinceDate,
        cast(floor(months_between(c.MonthEndDate, s.CustomerSinceDate)) as int) as TenureMonths,
        c.TotalPremium,
        c.NextRenewalDate,
        c.MonthsToNextRenewal,
        c.NextMotorRenewalDate,
        c.MotorMonthsToRenewal,
        13 - c.MotorMonthsToRenewal                                             as MotorPolicyMonth,
        c.AnyPolicyInRenewalWindow,
        c.MotorKeyCareHeld,
        c.MotorNcdProtected,
        c.MotorAddOnCount
from    (select  SCV_Customer_Key,
                 MonthEndDate,
                 MonthNo,
                 max(IsAnalysisMonth)                                                         as IsAnalysisMonth,
                 count(*)                                                                     as PoliciesHeld,
                 sum(case when PolicyTypeGroup = 'Motor' then 1 else 0 end)                   as MotorPolicies,
                 sum(case when PolicyTypeGroup = 'Home' then 1 else 0 end)                    as HomePolicies,
                 sum(case when PolicyTypeGroup = 'Van' then 1 else 0 end)                     as VanPolicies,
                 max(case when PolicyTypeGroup = 'Motor' then 1 else 0 end)                   as HoldsMotor,
                 max(case when PolicyTypeGroup = 'Home' then 1 else 0 end)                    as HoldsHome,
                 max(case when PolicyTypeGroup = 'Van' then 1 else 0 end)                     as HoldsVan,
                 sum(EURGrossPremClose)                                                       as TotalPremium,
                 min(RenewalDate)                                                             as NextRenewalDate,
                 min(MonthsToRenewal)                                                         as MonthsToNextRenewal,
                 min(case when PolicyTypeGroup = 'Motor' then RenewalDate end)                as NextMotorRenewalDate,
                 min(case when PolicyTypeGroup = 'Motor' then MonthsToRenewal end)            as MotorMonthsToRenewal,
                 max(InRenewalWindow)                                                         as AnyPolicyInRenewalWindow,
                 max(KeyCareHeld)                                                             as MotorKeyCareHeld,
                 max(NcdProtected)                                                            as MotorNcdProtected,
                 max(case when PolicyTypeGroup = 'Motor' and KeyCareHeld is not null and NcdProtected is not null
                          then KeyCareHeld + NcdProtected end)                                as MotorAddOnCount
         from    ${beh.target}.CustomerPolicyMonth
         group by SCV_Customer_Key, MonthEndDate, MonthNo) c
left join ${beh.stg}.beh_customer_since s on s.SCV_Customer_Key = c.SCV_Customer_Key


## Checks


In [ ]:
%%pyspark
# Row count, duplicate keys and null keys for every table this notebook built. Logged to BehBuildLog;
# the cell fails if any check failed, naming the table and the likely cause.
beh_check("BEH_10_Lifecycle", [{"table": "{stg}.beh_month_end", "key": ["MonthEndDate"]}, {"table": "{stg}.beh_policy_scv", "key": ["PolicyCode"]}, {"table": "{stg}.beh_customer_since", "key": ["SCV_Customer_Key"]}, {"table": "{stg}.beh_motor_addon", "key": ["PolicyCode", "MonthEndDate"]}, {"table": "{stg}.beh_policy_latest", "key": ["PolicyCode"]}, {"table": "{stg}.beh_motor_retention", "key": ["PolicyCode", "DueDate"]}, {"table": "{target}.CustomerPolicyMonth", "key": ["PolicyCode", "MonthEndDate"]}, {"table": "{target}.CustomerMonth", "key": ["SCV_Customer_Key", "MonthEndDate"]}])


In [ ]:
%%pyspark
beh_run_log("BEH_10_Lifecycle", "succeeded")
